# AI-Based Hiring Prediction System

## Project Objective
In this project, we build an AI-based Hiring Prediction System using a resume screening dataset.
The goal is to develop a Machine Learning model that predicts whether a candidate will be **Hired (1)** or **Rejected (0)** based on resume information such as skills, experience, education, certifications, projects, and salary expectations.

This project simulates a real-world AI resume screening system used in HR analytics.

## 1. Generating the Synthetic Dataset
Since the dataset is synthetic, we'll first generate it using Python. The dataset contains 1,000 resumes with the requested columns.

In [ ]:
import pandas as pd
import numpy as np
import random

np.random.seed(42)
random.seed(42)

n_samples = 1000

# Feature options
skills_pool = ['Python', 'SQL', 'Machine Learning', 'Data Analysis', 'Tableau', 'Excel', 'Java', 'C++', 'AWS', 'Azure', 'TensorFlow', 'PyTorch', 'Communication', 'Project Management', 'Agile']
education_levels = ['Bachelors', 'Masters', 'PhD', 'High School']
certifications_pool = ['AWS Certified', 'Azure Fundamentals', 'Google ML Engineer', 'PMP', 'Scrum Master', 'None']
job_roles = ['Data Scientist', 'Data Analyst', 'Software Engineer', 'Project Manager', 'ML Engineer']

data = {
    'Resume_ID': [f"RES_{i:04d}" for i in range(1, n_samples + 1)],
    'Name': [f"Candidate_{i}" for i in range(1, n_samples + 1)],
    'Skills': [", ".join(random.sample(skills_pool, k=random.randint(3, 7))) for _ in range(n_samples)],
    'Experience (Years)': np.random.randint(0, 15, size=n_samples),
    'Education': np.random.choice(education_levels, size=n_samples, p=[0.5, 0.3, 0.1, 0.1]),
    'Certifications': np.random.choice(certifications_pool, size=n_samples, p=[0.15, 0.15, 0.15, 0.1, 0.1, 0.35]),
    'Job Role': np.random.choice(job_roles, size=n_samples),
    'Salary Expectation ($)': np.random.randint(50000, 150000, size=n_samples),
    'Projects Count': np.random.randint(0, 10, size=n_samples)
}

df = pd.DataFrame(data)

# Generating Recruiter Decision (Target) based on some logic to simulate real-world hiring
# Higher experience, specific skills, and lower salary expectations increase hire chance
def hiring_logic(row):
    score = 0
    if row['Experience (Years)'] > 3: score += 2
    if 'Machine Learning' in row['Skills'] or 'Python' in row['Skills']: score += 2
    if row['Projects Count'] > 3: score += 1
    if row['Education'] in ['Masters', 'PhD']: score += 1
    if row['Certifications'] != 'None': score += 1
    if row['Salary Expectation ($)'] > 120000: score -= 2
    
    prob = 1 / (1 + np.exp(- (score - 3))) # Sigmoid function
    return 1 if random.random() < prob else 0

df['Recruiter Decision'] = df.apply(hiring_logic, axis=1)

# Save to CSV for reference
df.to_csv("synthetic_resumes.csv", index=False)
df.head()

## 2. Exploratory Data Analysis (EDA)
Let's explore the dataset to understand the features and the target distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Recruiter Decision', palette='Set2')
plt.title('Target Variable Distribution (Recruiter Decision)')
plt.show()

### Feature Distributions
Let's examine how Experience and Salary Expectations relate to the Recruiter Decision.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Recruiter Decision', y='Experience (Years)', ax=axes[0], palette='Set2')
axes[0].set_title('Experience (Years) vs Hiring Decision')

sns.boxplot(data=df, x='Recruiter Decision', y='Salary Expectation ($)', ax=axes[1], palette='Set2')
axes[1].set_title('Salary Expectation vs Hiring Decision')

plt.tight_layout()
plt.show()

## 3. Data Preprocessing & Feature Engineering
We need to handle text features like `Skills`, categorical features like `Education` and `Certifications`, and numerical features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Separate features and target
X = df.drop(columns=['Resume_ID', 'Name', 'Recruiter Decision'])
y = df['Recruiter Decision']

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Custom Transformer for comma-separated skills using CountVectorizer
from sklearn.base import BaseEstimator, TransformerMixin

class SkillTransformerDF(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.vectorizer = CountVectorizer(tokenizer=lambda x: [s.strip() for s in x.split(',')], lowercase=False, token_pattern=None)
        
    def fit(self, X, y=None):
        self.vectorizer.fit(X.iloc[:, 0])
        return self
        
    def transform(self, X):
        return self.vectorizer.transform(X.iloc[:, 0]).toarray()
    
    def get_feature_names_out(self, input_features=None):
        return [f"skill_{f}" for f in self.vectorizer.get_feature_names_out()]

# Preprocessing Pipeline
numeric_features = ['Experience (Years)', 'Salary Expectation ($)', 'Projects Count']
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_features = ['Education', 'Certifications', 'Job Role']
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('skills', SkillTransformerDF(), ['Skills'])
    ])

## 4. Model Training
We will train a Logistic Regression model and a Random Forest model to compare their performance.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Model 1: Logistic Regression
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', LogisticRegression(random_state=42, class_weight='balanced'))])
lr_pipeline.fit(X_train, y_train)

# Model 2: Random Forest
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced', n_estimators=100))])
rf_pipeline.fit(X_train, y_train)


## 5. Model Evaluation
We will evaluate the models using Accuracy, Precision, Recall, F1-Score, and visualize the Confusion Matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay

def evaluate_model(model, name, X_test, y_test):
    y_pred = model.predict(X_test)
    print(f"=== {name} Evaluation ===")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Rejected', 'Hired'])
    disp.plot(cmap='Blues', values_format='d')
    plt.title(f'{name} - Confusion Matrix')
    plt.show()

evaluate_model(lr_pipeline, "Logistic Regression", X_test, y_test)
evaluate_model(rf_pipeline, "Random Forest", X_test, y_test)


## 6. Conclusion and Key Insights

### Data Analysis Key Findings
- **Experience Matters:** Candidates with more years of experience show a significantly higher likelihood of being hired.
- **Salary Constraints:** Very high salary expectations negatively impact the hiring probability.
- **Data Balance:** Using a realistic simulation, the generated dataset has a logical distribution between Hired and Rejected candidates.

### Next Steps & Model Selection
- The **Logistic Regression** model and **Random Forest** model both perform well, learning the underlying logic we used to generate the dataset. 
- Random Forest often captures non-linear relationships (like specific skill combinations paired with high experience) better than Logistic Regression.
- For a real-world HR system, **Interpretability** is crucial to avoid bias, making Logistic Regression a very strong candidate. 
- **Next steps** could include deploying this pipeline into a Flask/FastAPI service or saving the model using `joblib` for integration into an applicant tracking system.